In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Lasso, LogisticRegression
from sklearn.metrics import mean_squared_error, f1_score

import matplotlib.pyplot as plt
%matplotlib inline

import joblib
import scipy.stats as stats

In [18]:
data = pd.read_csv('resources/dataset.csv')
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 28 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   row_id             500000 non-null  int64  
 1   user_id            500000 non-null  int64  
 2   split              500000 non-null  str    
 3   day_index          500000 non-null  int64  
 4   day_of_week        500000 non-null  int64  
 5   is_weekend         500000 non-null  int64  
 6   is_school_holiday  500000 non-null  int64  
 7   minute_of_day      500000 non-null  int64  
 8   hour_sin           500000 non-null  float64
 9   hour_cos           500000 non-null  float64
 10  age                500000 non-null  int64  
 11  gender             500000 non-null  str    
 12  occupation         500000 non-null  str    
 13  chronotype         500000 non-null  str    
 14  lives_with_others  500000 non-null  int64  
 15  object             500000 non-null  str    
 16  object_conf  

## Unnecessary Features

- user_id
- row_id
- split
- gender
- chronotype
- light_level
- action_conf
- object_conf
- lives_with_others

## Target
- event

## Numerical
- minute_of_day
- hour_sin
- hour_cos
- age
- dwell_seconds


## Categorical
- day_of_week
- is_weekend
- is_school_holiday
- occupation
- object
- action
- prev_action
- object_in_hand
- zone
- is_off_routine


In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class AttributesOrganizer(BaseEstimator, TransformerMixin):
    def __init__(self, add_ratio=True):
        self.add_ratio = add_ratio

    def fit(self, X, y=None):
        return self  # nothing else to do

    def transform(self, X):

        X = data.copy()
        X = X.drop([
                    'user_id', 
                'row_id', 
                'split', 
                'gender', 
                'chronotype', 
                'light_level', 
                'chronotype', 
                'action_conf', 
                'object_conf', 
                'is_hard_negative',
                'lives_with_others',
                'event'], axis=1)




In [32]:
target = data['event']
X_train, X_test, t_train, t_test = train_test_split(X, target, train_size=0.8, random_state=42, shuffle=True)

X_train.shape, X_test.shape, t_train.shape, t_test.shape

((400000, 17), (100000, 17), (400000,), (100000,))

In [33]:
from sklearn.preprocessing import FunctionTransformer

num_scale = ["age"]
num_log   = ["dwell_seconds"]
num_pass  = ["hour_sin", "hour_cos"]
binary    = ["is_weekend", "is_school_holiday", "is_off_routine"]
cat       = ["day_of_week", "occupation", "object", "action",
             "prev_action", "object_in_hand", "zone"]

pre = ColumnTransformer([
    ("scale", StandardScaler(), num_scale),
    ("log",   Pipeline([("log", FunctionTransformer(np.log1p)),
                        ("sc",  StandardScaler())]), num_log),
    ("pass",  "passthrough", num_pass + binary),
    ("ohe",   OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat),
])


full_pipeline = Pipeline([('attribs_organizer', AttributesOrganizer()),
                                  ('encod_and_scale', pre)])
full_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('attribs_organizer', ...), ('encod_and_scale', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,add_ratio,True
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('scale', ...), ('log', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold spar

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score, recall_score, classification_report

random_forest = RandomForestClassifier(n_jobs=-1, random_state=42)
param_grid = {'n_estimators': [10, 50, 100, 200, 300], 'max_depth': [10, 20, 30]}

search = GridSearchCV(random_forest, param_grid=param_grid, scoring='f1', n_jobs=-1, cv=gro)